# HyLeakAI — subsurface site-suitability screening

No GPU, no internet. Reads `constants.npy` / `states.npy` from the
**hyleakai-dataset-download-conversion** kernel's output (attached below as
an input, so nothing is re-downloaded), computes per-realisation geology and
caprock-margin features using the same code already tested in
`src/leakage/`, and clusters the 1,000 realisations into suitability groups.

**Before running:** *Add Input -> Notebook Output Files ->
sonilnegi/hyleakai-dataset-download-conversion*, so `/kaggle/input/` has the
converted arrays.

In [ ]:
# Locate the converted arrays wherever Kaggle mounted the input kernel.
from pathlib import Path

candidates = list(Path("/kaggle/input").rglob("constants.npy"))
assert candidates, (
    "constants.npy not found under /kaggle/input. Add Input -> Notebook Output "
    "Files -> sonilnegi/hyleakai-dataset-download-conversion, then re-run.\n"
    f"Currently mounted: {list(Path('/kaggle/input').iterdir())}"
)
DATA = candidates[0].parent
print("Using data from:", DATA)
print("contents:", sorted(p.name for p in DATA.iterdir()))

In [ ]:
# Recreate the src/ package layout this notebook needs (mirrors src/leakage/
# in the repo -- these are the exact tested files, embedded verbatim).
from pathlib import Path
Path("src/leakage").mkdir(parents=True, exist_ok=True)
Path("src/__init__.py").touch()
Path("src/leakage/__init__.py").touch()

In [ ]:
%%writefile src/config.py
"""Central configuration for HyLeakAI.

Every constant here is tagged with its provenance:

  [DATASET]  Read directly from the Mao et al. (2025) dataset or stated in the paper.
             These are facts.
  [DERIVED]  Computed from a [DATASET] value by an explicit, stated calculation.
  [ASSUMED]  Not present in the paper or dataset. Our choice, with a justification.
             Every one of these must be sweepable and must be disclosed in any writeup.

Reference:
  Mao, S., Carbonero, A., & Mehana, M. (2025). Deep learning for subsurface flow:
  A comparative study of U-Net, Fourier neural operators, and transformers in
  underground hydrogen storage. JGR: Machine Learning and Computation, 2,
  e2024JH000401. https://doi.org/10.1029/2024JH000401
  Dataset: https://zenodo.org/records/14029514 (CC-BY-4.0 / MIT)
"""

from __future__ import annotations

from dataclasses import dataclass, field
from pathlib import Path

# --------------------------------------------------------------------------
# Paths
# --------------------------------------------------------------------------

REPO_ROOT = Path(__file__).resolve().parent.parent
DATA_DIR = REPO_ROOT / "data"
RAW_LMDB = DATA_DIR / "data.mdb"          # 12.38 GB download from Zenodo
CONSTANTS_NPY = DATA_DIR / "constants.npy"  # (1000, 2, 128, 128) float16
STATES_NPY = DATA_DIR / "states.npy"        # (1000, 60, 2, 128, 128) float16
STATS_JSON = DATA_DIR / "stats.json"
CHECKPOINT_DIR = REPO_ROOT / "checkpoints"
OUTPUT_DIR = REPO_ROOT / "outputs"

# Channel ordering, used consistently everywhere.
CONST_POROSITY, CONST_PERMEABILITY = 0, 1
STATE_PRESSURE, STATE_SATURATION, STATE_AUX = 0, 1, 2
N_STATE_CHANNELS = 3

# [DATASET, undocumented] The Zenodo record and the repository README both
# describe the per-timestep value as a 2-tuple (pressure, H2 saturation). It is
# actually a 3-tuple. We measured the third channel across timesteps and
# simulations:
#
#     corr(aux, pressure) = -0.993 to -0.999 at every timestep
#     aux / (P - P_init)  = -1.17e-4 per bar, constant across time
#
# That coefficient is a compressibility (1.17e-9 /Pa), the right order for
# brine plus pore compressibility, so the field is most consistent with a
# saturation- or volume-deviation driven by pressure. Roughly 10% of its
# variance is not explained by pressure and tracks the H2 plume.
#
# We cannot name it definitively, so we do not pretend to. It is preserved in
# the converted arrays under a neutral name, and it is NOT used as a model
# target: the paper predicts pressure and H2 saturation, and those are what we
# reproduce. Being ~99% collinear with pressure, it carries little independent
# information anyway.
STATE_CHANNEL_NAMES = ("pressure", "saturation", "aux_undocumented")


# --------------------------------------------------------------------------
# Grid and reservoir geometry
# --------------------------------------------------------------------------

GRID = 128                      # [DATASET] 128 x 128 x 1 cells
DOMAIN_M = 7680.0               # [DATASET] 7,680 m length and width
THICKNESS_M = 100.0             # [DATASET] 100 m reservoir thickness
CELL_M = DOMAIN_M / GRID        # [DERIVED] 60.0 m cell edge
CELL_AREA_M2 = CELL_M**2        # [DERIVED] 3,600 m^2
CELL_VOLUME_M3 = CELL_AREA_M2 * THICKNESS_M  # [DERIVED] 3.6e5 m^3

# [DATASET] "no-flow boundaries at the top and bottom, and outflow boundaries on
# the sides". Top/bottom = caprock/baserock (sealed). The SIDES are open, so
# lateral H2 migration to the domain edge is genuine escape from the model.
# This is the only leakage signal actually present in the simulation.
LATERAL_BOUNDARY_IS_OUTFLOW = True
CAPROCK_IS_SEALED_IN_SIM = True

# [ASSUMED] Well is described only as "a central well". On a 128-cell axis the
# centre falls between indices 63 and 64, so we take the 2x2 central block.
# validate_well_location() in src/data/explore.py confirms this empirically by
# locating the pressure maximum during an injection step.
WELL_IJ = (63, 63)              # top-left of the central 2x2 block
WELL_BLOCK = (slice(63, 65), slice(63, 65))
GRID_CENTER = (GRID - 1) / 2.0  # 63.5, used for the symmetric distance map


# --------------------------------------------------------------------------
# Temporal structure
# --------------------------------------------------------------------------

N_SIMS = 1000                   # [DATASET]
N_TIMESTEPS = 60                # [DATASET] usable steps; stored t=1..60
MONTHS_PER_STEP = 2             # [DATASET] output every two months
STEPS_PER_CYCLE = 6             # [DERIVED] 12 months / 2
N_CYCLES = 10                   # [DATASET] 10 annual storage cycles
INJECTION_STEPS_PER_CYCLE = 3   # [DERIVED] 6-month injection stage
# [DATASET] t=0 is the pre-injection state and is byte-identical across all
# 1,000 simulations. We drop it during conversion; keeping it would add 1,000
# duplicate samples that teach the model nothing.
DROP_TIMESTEP_ZERO = True


def cycle_index(t: int) -> int:
    """Cyclic index for timestep t (1-based): +1 injection, -1 withdrawal.

    [DATASET] Paper section 3.4: "the model receives a value of 1 during the
    injection stage and -1 during the withdrawal stage". Each annual cycle is a
    6-month injection followed by a 6-month withdrawal, i.e. 3 steps of each.
    """
    return 1 if ((t - 1) % STEPS_PER_CYCLE) < INJECTION_STEPS_PER_CYCLE else -1


def cycle_number(t: int) -> int:
    """1-based storage-cycle number (1..10) for timestep t (1-based)."""
    return (t - 1) // STEPS_PER_CYCLE + 1


def is_injection(t: int) -> bool:
    return cycle_index(t) == 1


# --------------------------------------------------------------------------
# Pressure regime
# --------------------------------------------------------------------------

P_INIT_BAR = 197.2              # [DATASET] "initial reservoir pressure (197.2e5 Pa)"
BAR_TO_PA = 1.0e5

# [ASSUMED] Reservoir depth is never stated. We infer it by assuming the
# reservoir is initially at hydrostatic pressure, which is standard for a
# depleted-then-repressurised gas reservoir, using a brine gradient of
# 0.105 bar/m (~1,050 kg/m^3). This yields ~1,878 m, a typical depleted gas
# reservoir depth. Only used for the caprock fracture-pressure criterion (T2).
HYDROSTATIC_GRADIENT_BAR_PER_M = 0.105
RESERVOIR_DEPTH_M = P_INIT_BAR / HYDROSTATIC_GRADIENT_BAR_PER_M  # ~1878 m


# --------------------------------------------------------------------------
# Leakage-label physics  (Phase 3)
# --------------------------------------------------------------------------


@dataclass
class LeakageConfig:
    """Physical constants for the derived leakage labels.

    None of these come from the dataset. They are documented modelling choices,
    and the defaults are mid-range values from the UHS / CO2-storage literature.
    Everything here should be swept in a sensitivity analysis before any number
    from it is reported as a result.
    """

    # --- T1: lateral containment loss (the one REAL signal) ---
    # Width in cells of the outer ring treated as "at the outflow boundary".
    # 4 cells = 240 m.
    boundary_ring_cells: int = 4
    # Saturation above which a cell counts as containing mobile H2.
    plume_saturation_threshold: float = 0.05

    # --- T2: caprock breach criterion (physics criterion, not a flux) ---
    # [ASSUMED] Fracture gradient. Typical sedimentary basin range is
    # 0.15-0.20 bar/m; 0.17 is a common screening default.
    frac_gradient_bar_per_m: float = 0.17
    frac_gradient_sweep: tuple[float, ...] = (0.15, 0.17, 0.20)

    # --- T3: semi-analytical fault conduit leakage (primary XGBoost target) ---
    # [ASSUMED] Caprock thickness — the vertical path length of the conduit.
    caprock_thickness_m: float = 50.0
    # [ASSUMED] H2 dynamic viscosity at ~200 bar and reservoir temperature.
    # Hydrogen is ~0.9-1.1e-5 Pa.s over this range; we take 9.5e-6 Pa.s.
    h2_viscosity_pa_s: float = 9.5e-6
    # [ASSUMED] Corey relative-permeability parameters for the gas phase.
    corey_swr: float = 0.20     # irreducible water saturation
    corey_sgr: float = 0.05     # residual (immobile) gas saturation
    corey_ng: float = 2.0       # Corey exponent

    # Monte-Carlo fault sampling. Log-uniform on permeability because it spans
    # three orders of magnitude.
    n_faults_per_sim: int = 20
    fault_perm_m2_range: tuple[float, float] = (1e-15, 1e-12)   # ~1-1000 mD
    fault_length_m_range: tuple[float, float] = (200.0, 2000.0)
    fault_width_m_range: tuple[float, float] = (1.0, 10.0)      # damage-zone width
    # Faults are placed within this radial band around the well, in metres, so
    # that the sample spans "right at the plume" to "far outside it".
    fault_radius_m_range: tuple[float, float] = (200.0, 3000.0)
    rng_seed: int = 20260809

    @property
    def frac_pressure_bar(self) -> float:
        """[DERIVED] Caprock fracture pressure at reservoir depth."""
        return RESERVOIR_DEPTH_M * self.frac_gradient_bar_per_m


LEAKAGE = LeakageConfig()


# --------------------------------------------------------------------------
# Model / training  (Phase 2)
# --------------------------------------------------------------------------


@dataclass
class UNetConfig:
    """U-Net-Small from the paper, Table 1: depth 4, embedding 32, ~7.7M params.

    Rationale for Small over Large: the paper reports U-Net-Small WITH cyclic and
    distance information reaching 8.6% pressure test error, level with
    U-Net-Large's 8.61% at 124M parameters and 35 GB. Without those two inputs
    Small degrades to 32.7%. So the extra input channels, not the parameter
    count, are what buy the accuracy here.
    """

    depth: int = 4
    embedding: int = 32
    in_channels: int = 5     # porosity, permeability, time, cyclic, distance
    out_channels: int = 2    # pressure, saturation
    use_cyclic: bool = True
    use_distance: bool = True


@dataclass
class TrainConfig:
    """Optimiser settings from the paper's Appendix Table A1."""

    batch_size: int = 32          # paper used 128 on a 40 GB A100; 32 fits a T4
    learning_rate: float = 1e-4
    weight_decay: float = 1e-5
    lr_halve_every: int = 50      # "halved every 50 epochs"
    epochs: int = 120
    # Multi-head loss weights. The paper trains separate models per state
    # variable; we use one two-headed model to halve training cost. Saturation
    # lives in [0,1] and pressure is standardised, so relative-L2 on each head
    # is already scale-free and equal weights are a sound starting point.
    lambda_saturation: float = 1.0
    lambda_pressure: float = 1.0
    num_workers: int = 4
    seed: int = 20260809


# --------------------------------------------------------------------------
# Data splits — BY SIMULATION, never by sample
# --------------------------------------------------------------------------

# [DATASET] Paper section 3.3: 700 train / 150 validation / 150 test, assigned by
# randomly shuffling the 1,000 simulations. All 60 timesteps of a simulation
# follow its simulation into the same split. Splitting by sample instead would
# put timestep t of a simulation in train and t+1 in test, which leaks almost
# everything and inflates accuracy.
SPLIT_SIZES = {"train": 700, "val": 150, "test": 150}
SPLIT_SEED = 20260809


def simulation_splits(seed: int = SPLIT_SEED) -> dict[str, list[int]]:
    """Deterministic simulation-level split. Used by BOTH the U-Net and XGBoost
    stages so a simulation never appears in one stage's training set and
    another's test set."""
    import numpy as np

    rng = np.random.default_rng(seed)
    ids = rng.permutation(N_SIMS)
    n_tr, n_va = SPLIT_SIZES["train"], SPLIT_SIZES["val"]
    return {
        "train": sorted(ids[:n_tr].tolist()),
        "val": sorted(ids[n_tr : n_tr + n_va].tolist()),
        "test": sorted(ids[n_tr + n_va :].tolist()),
    }

In [ ]:
%%writefile src/leakage/labels.py
"""Leakage targets derived from the UHS simulations.

READ THIS BEFORE USING ANY NUMBER OUT OF THIS MODULE.

The Mao et al. dataset contains porosity, permeability, pressure and H2
saturation. It contains no faults, no caprock properties, no fluxes and no
leakage labels. We have no reservoir simulator, so we cannot produce true
simulated leakage flux. Rather than invent one and call it ground truth, this
module produces three targets and labels each one for what it actually is:

  T1  lateral containment loss   REAL SIGNAL IN THE DATA
      The paper states the simulation has "no-flow boundaries at the top and
      bottom, and outflow boundaries on the sides". H2 that reaches the lateral
      boundary genuinely leaves the model domain. Nothing is assumed here
      beyond where we draw the boundary ring.

  T2  caprock breach margin      PHYSICS CRITERION, NOT A FLUX
      Pressure-based screening of the kind used routinely in CO2 and H2 storage
      risk assessment, applied to the simulated pressure field. It says
      "pressure exceeded a fracture threshold", not "hydrogen leaked". The
      caprock is a sealed no-flow boundary in these simulations, so no vertical
      leakage was ever simulated.

  T3  fault conduit leakage      SEMI-ANALYTICAL MODEL, DERIVED LABEL
      Darcy flux through a HYPOTHETICAL fault overlaid on the simulated
      pressure and saturation fields. This is the primary XGBoost target
      because it is the only one where fault features are causally connected to
      the label, which SHAP attributions require in order to mean anything.

Why overlaying a fault is not circular: the fault is not an input to the U-Net.
The U-Net predicts the flow field from geology alone; the fault is introduced
afterwards. The system therefore answers "given this fault hypothesis, how bad
is it?", which supports Monte-Carlo over unknown fault properties in
milliseconds. That is a real capability, not a restatement of the simulator.

Every assumed constant lives in `config.LeakageConfig` and is swept in
`sensitivity.py`. None of them come from the dataset.
"""

from __future__ import annotations

from dataclasses import dataclass, asdict

import numpy as np

from src import config as C
from src.config import LeakageConfig, LEAKAGE


# ==========================================================================
# T1 — Lateral containment loss  (the one real signal)
# ==========================================================================


def boundary_ring_mask(grid: int = C.GRID, ring_cells: int = None) -> np.ndarray:
    """Cells within `ring_cells` of the open lateral boundary."""
    ring_cells = ring_cells if ring_cells is not None else LEAKAGE.boundary_ring_cells
    mask = np.zeros((grid, grid), dtype=bool)
    mask[:ring_cells, :] = True
    mask[-ring_cells:, :] = True
    mask[:, :ring_cells] = True
    mask[:, -ring_cells:] = True
    return mask


def h2_pore_volume(porosity: np.ndarray, saturation: np.ndarray) -> np.ndarray:
    """H2 pore volume per cell, m^3.

    Deliberately a pore volume rather than a mass: converting to mass needs an
    H2 density from a real-gas equation of state at a reservoir temperature the
    paper never states. Pore volume is exact given the data; mass would smuggle
    in another assumption for no analytical gain.
    """
    return porosity * saturation * C.CELL_VOLUME_M3


def lateral_containment(
    porosity: np.ndarray,
    saturation: np.ndarray,
    cfg: LeakageConfig = LEAKAGE,
    ring: np.ndarray | None = None,
) -> dict:
    """T1: how much H2 has migrated into the open-boundary ring."""
    ring = boundary_ring_mask(saturation.shape[-1], cfg.boundary_ring_cells) if ring is None else ring
    hpv = h2_pore_volume(porosity, saturation)
    total = float(hpv.sum())
    in_ring = float(hpv[ring].sum())
    s_ring_max = float(saturation[ring].max())
    return {
        "hpv_total_m3": total,
        "hpv_ring_m3": in_ring,
        "hpv_ring_fraction": in_ring / total if total > 0 else 0.0,
        "boundary_saturation_max": s_ring_max,
        "lateral_breach": bool(s_ring_max > cfg.plume_saturation_threshold),
    }


def t1_viability_scan(
    constants: np.ndarray,
    states: np.ndarray,
    cfg: LeakageConfig = LEAKAGE,
    progress: bool = True,
) -> dict:
    """GO / NO-GO for T1.

    With one central well in a 7,680 m domain, the plume may never reach the
    outflow boundary within 10 years. If it does not, T1 is identically zero
    everywhere and is useless as a label. This must be run and reported before
    T1 is used for anything; the honest outcome may well be "T1 is dropped".
    """
    from src.data.lmdb_convert import restore_pressure  # noqa: F401  (kept for symmetry)

    n_sims, n_steps = states.shape[0], states.shape[1]
    ring = boundary_ring_mask(states.shape[-1], cfg.boundary_ring_cells)

    per_sim_max = np.zeros(n_sims, dtype=np.float64)
    max_ring_fraction = 0.0
    first_breach_step = np.full(n_sims, -1, dtype=np.int32)

    for sim in range(n_sims):
        poro = np.asarray(constants[sim, C.CONST_POROSITY], np.float32)
        sat = np.asarray(states[sim, :, C.STATE_SATURATION], np.float32)  # (T, H, W)
        ring_sat = sat[:, ring]                      # (T, n_ring_cells)
        step_max = ring_sat.max(axis=1)              # (T,)
        per_sim_max[sim] = float(step_max.max())

        breached = np.nonzero(step_max > cfg.plume_saturation_threshold)[0]
        if breached.size:
            first_breach_step[sim] = int(breached[0]) + 1  # timesteps are 1-based
            hpv = poro[None] * sat * C.CELL_VOLUME_M3
            frac = hpv[:, ring].sum(axis=1) / np.maximum(
                hpv.reshape(n_steps, -1).sum(axis=1), 1e-30
            )
            max_ring_fraction = max(max_ring_fraction, float(frac.max()))

        if progress and (sim + 1) % 100 == 0:
            print(f"  scanned {sim + 1}/{n_sims} simulations", flush=True)

    n_breach = int((first_breach_step > 0).sum())
    viable = n_breach >= max(10, int(0.01 * n_sims))
    return {
        "n_sims": n_sims,
        "threshold": cfg.plume_saturation_threshold,
        "ring_cells": cfg.boundary_ring_cells,
        "n_sims_with_breach": n_breach,
        "breach_rate": n_breach / n_sims,
        "boundary_saturation_max_overall": float(per_sim_max.max()),
        "boundary_saturation_mean_of_sim_max": float(per_sim_max.mean()),
        "max_ring_hpv_fraction": max_ring_fraction,
        "earliest_breach_timestep": (
            int(first_breach_step[first_breach_step > 0].min()) if n_breach else None
        ),
        "T1_VIABLE": bool(viable),
        "verdict": (
            f"T1 usable: {n_breach}/{n_sims} simulations reach the open boundary."
            if viable
            else (
                f"T1 NOT usable: only {n_breach}/{n_sims} simulations show boundary "
                f"saturation above {cfg.plume_saturation_threshold}. The plume stays "
                f"well inside the 7,680 m domain, so lateral containment loss carries "
                f"no signal. Drop T1 and say so; T2 and T3 carry the project."
            )
        ),
    }


# ==========================================================================
# T2 — Caprock breach margin  (criterion, not a flux)
# ==========================================================================


def caprock_margin(pressure_bar: np.ndarray, cfg: LeakageConfig = LEAKAGE) -> dict:
    """T2: how close the field is to the assumed caprock fracture pressure.

    margin = (P_max - P_initial) / (P_frac - P_initial)

    0 means "at the initial reservoir pressure", 1 means "at the fracture
    pressure". P_frac comes from an ASSUMED depth and fracture gradient; sweep
    `frac_gradient_bar_per_m` before reporting anything from this.
    """
    p_frac = cfg.frac_pressure_bar
    p_max = float(np.max(pressure_bar))
    denom = p_frac - C.P_INIT_BAR
    margin = (p_max - C.P_INIT_BAR) / denom if denom > 0 else np.inf
    return {
        "p_max_bar": p_max,
        "p_frac_bar": p_frac,
        "caprock_margin": margin,
        "caprock_exceeded": bool(margin > 1.0),
    }


# ==========================================================================
# T3 — Semi-analytical fault conduit leakage  (primary target)
# ==========================================================================


@dataclass
class Fault:
    """A hypothetical vertical fault cutting the caprock, as a surface trace."""

    fault_id: int
    x_m: float           # trace centre, metres from the domain's lower-left corner
    y_m: float
    length_m: float
    width_m: float       # damage-zone width; sets the conduit's cross-section
    permeability_m2: float
    orientation_rad: float

    @property
    def area_m2(self) -> float:
        """Conduit cross-section presented to vertical flow."""
        return self.length_m * self.width_m

    def as_dict(self) -> dict:
        d = asdict(self)
        d["area_m2"] = self.area_m2
        return d


def sample_faults(
    n: int, cfg: LeakageConfig = LEAKAGE, seed: int | None = None
) -> list[Fault]:
    """Monte-Carlo fault realisations.

    Permeability is log-uniform because it spans three orders of magnitude
    (~1 to ~1000 mD) and a uniform draw would put almost all mass at the
    high end. Faults are placed on a radial band around the central well so the
    sample spans "cutting straight through the plume" to "well outside it" —
    without that spread, plume-to-fault distance would carry no information and
    the whole feature set would collapse.
    """
    rng = np.random.default_rng(cfg.rng_seed if seed is None else seed)
    faults = []
    for i in range(n):
        radius = rng.uniform(*cfg.fault_radius_m_range)
        bearing = rng.uniform(0, 2 * np.pi)
        centre = C.DOMAIN_M / 2.0
        faults.append(
            Fault(
                fault_id=i,
                x_m=centre + radius * np.cos(bearing),
                y_m=centre + radius * np.sin(bearing),
                length_m=rng.uniform(*cfg.fault_length_m_range),
                width_m=rng.uniform(*cfg.fault_width_m_range),
                permeability_m2=float(
                    10 ** rng.uniform(*np.log10(cfg.fault_perm_m2_range))
                ),
                orientation_rad=rng.uniform(0, np.pi),
            )
        )
    return faults


def distance_to_fault_field(fault: Fault, grid: int = C.GRID) -> np.ndarray:
    """Distance in metres from every cell centre to the fault trace segment.

    Standard point-to-segment distance, vectorised over the grid. Computed once
    per fault and reused across all 60 timesteps.
    """
    coords = (np.arange(grid) + 0.5) * C.CELL_M
    yy, xx = np.meshgrid(coords, coords, indexing="ij")

    half = fault.length_m / 2.0
    dx, dy = np.cos(fault.orientation_rad), np.sin(fault.orientation_rad)
    ax, ay = fault.x_m - half * dx, fault.y_m - half * dy
    bx, by = fault.x_m + half * dx, fault.y_m + half * dy

    vx, vy = bx - ax, by - ay
    seg_len_sq = vx * vx + vy * vy
    if seg_len_sq <= 0:
        return np.hypot(xx - ax, yy - ay)

    t = np.clip(((xx - ax) * vx + (yy - ay) * vy) / seg_len_sq, 0.0, 1.0)
    return np.hypot(xx - (ax + t * vx), yy - (ay + t * vy))


def fault_cell_mask(distance_field: np.ndarray) -> np.ndarray:
    """Cells the trace passes through. The 0.75-cell radius keeps the rasterised
    trace connected at any orientation."""
    mask = distance_field <= C.CELL_M * 0.75
    if not mask.any():  # a very short trace can miss every cell centre
        mask = distance_field == distance_field.min()
    return mask


def corey_krg(saturation, cfg: LeakageConfig = LEAKAGE):
    """Corey gas relative permeability.

    k_rg = ((S_g - S_gr) / (1 - S_wr - S_gr)) ** n_g, clipped to [0, 1].

    This is what makes the label physically sensible rather than a pressure
    proxy: below residual gas saturation the H2 is immobile and cannot leak at
    all, no matter how high the pressure. Without it, a fault far from the plume
    would still "leak" whenever the field pressurised.
    """
    sg = np.asarray(saturation, dtype=np.float64)
    denom = 1.0 - cfg.corey_swr - cfg.corey_sgr
    if denom <= 0:
        raise ValueError("corey_swr + corey_sgr must be < 1")
    se = np.clip((sg - cfg.corey_sgr) / denom, 0.0, 1.0)
    return se**cfg.corey_ng


def fault_leakage_flux(
    pressure_bar: np.ndarray,
    saturation: np.ndarray,
    fault: Fault,
    cfg: LeakageConfig = LEAKAGE,
    fault_mask: np.ndarray | None = None,
    distance_field: np.ndarray | None = None,
) -> dict:
    """T3: Darcy flux of H2 up a hypothetical fault, m^3/s.

        Q = (k_f * k_rg * A_f / mu_H2) * dP / L_caprock

    The driving force `dP` is the overpressure above the initial reservoir
    pressure, which we take to be hydrostatic. Two consequences, both intended:

      * During withdrawal the field drops below its initial pressure, dP is
        clipped to zero and no upward leakage occurs. Fluid would flow the other
        way, which is not leakage.
      * Leakage requires BOTH overpressure and mobile H2 at the fault. A
        pressurised fault with no plume on it leaks nothing.

    Units: m^2 * m^2 * Pa / (Pa.s * m) = m^3/s.
    """
    if distance_field is None:
        distance_field = distance_to_fault_field(fault, pressure_bar.shape[-1])
    if fault_mask is None:
        fault_mask = fault_cell_mask(distance_field)

    p_fault = float(np.mean(pressure_bar[fault_mask]))
    s_fault = float(np.mean(saturation[fault_mask]))
    krg = float(corey_krg(s_fault, cfg))

    overpressure_pa = max(p_fault - C.P_INIT_BAR, 0.0) * C.BAR_TO_PA
    q = (
        fault.permeability_m2
        * krg
        * fault.area_m2
        / cfg.h2_viscosity_pa_s
        * overpressure_pa
        / cfg.caprock_thickness_m
    )

    plume = saturation > cfg.plume_saturation_threshold
    d_plume = float(distance_field[plume].min()) if plume.any() else np.nan

    return {
        "q_fault_m3_s": q,
        "p_fault_bar": p_fault,
        "s_fault": s_fault,
        "krg_fault": krg,
        "overpressure_bar": overpressure_pa / C.BAR_TO_PA,
        "distance_plume_to_fault_m": d_plume,
        "fault_perm_m2": fault.permeability_m2,
        "fault_area_m2": fault.area_m2,
        "fault_length_m": fault.length_m,
    }


# ==========================================================================
# Self-checks
# ==========================================================================


def check_monotonicity(cfg: LeakageConfig = LEAKAGE) -> bool:
    """The label must respond to its drivers in the physically correct
    direction. If it does not, every SHAP attribution built on it is noise."""
    grid = C.GRID
    ok = True

    # A pressurised field with a plume covering the middle of the domain.
    pressure = np.full((grid, grid), C.P_INIT_BAR + 30.0, np.float32)
    saturation = np.zeros((grid, grid), np.float32)
    saturation[40:88, 40:88] = 0.6

    base = Fault(0, C.DOMAIN_M / 2, C.DOMAIN_M / 2, 1000.0, 5.0, 1e-13, 0.0)

    # 1. Higher fault permeability must not reduce flux.
    qs = [
        fault_leakage_flux(
            pressure, saturation,
            Fault(0, base.x_m, base.y_m, base.length_m, base.width_m, k, 0.0), cfg
        )["q_fault_m3_s"]
        for k in (1e-15, 1e-14, 1e-13, 1e-12)
    ]
    mono_k = all(b >= a for a, b in zip(qs, qs[1:]))
    print(f"{'OK  ' if mono_k else 'FAIL'} flux increases with fault permeability: "
          f"{[f'{q:.3e}' for q in qs]}")
    ok &= mono_k

    # 2. Higher overpressure must not reduce flux.
    qs = [
        fault_leakage_flux(np.full((grid, grid), C.P_INIT_BAR + dp, np.float32),
                           saturation, base, cfg)["q_fault_m3_s"]
        for dp in (0.0, 10.0, 30.0, 60.0)
    ]
    mono_p = all(b >= a for a, b in zip(qs, qs[1:]))
    print(f"{'OK  ' if mono_p else 'FAIL'} flux increases with overpressure: "
          f"{[f'{q:.3e}' for q in qs]}")
    ok &= mono_p

    # 3. Withdrawal (pressure below initial) must produce exactly zero.
    q_draw = fault_leakage_flux(
        np.full((grid, grid), C.P_INIT_BAR - 40.0, np.float32), saturation, base, cfg
    )["q_fault_m3_s"]
    zero_draw = q_draw == 0.0
    print(f"{'OK  ' if zero_draw else 'FAIL'} no upward leakage during withdrawal: "
          f"Q = {q_draw:.3e}")
    ok &= zero_draw

    # 4. Immobile H2 (below residual saturation) must produce exactly zero.
    q_immobile = fault_leakage_flux(
        pressure, np.full((grid, grid), cfg.corey_sgr * 0.5, np.float32), base, cfg
    )["q_fault_m3_s"]
    zero_immobile = q_immobile == 0.0
    print(f"{'OK  ' if zero_immobile else 'FAIL'} no leakage below residual gas "
          f"saturation: Q = {q_immobile:.3e}")
    ok &= zero_immobile

    # 5. A fault far from the plume must leak less than one cutting through it.
    q_near = fault_leakage_flux(pressure, saturation, base, cfg)
    far = Fault(1, C.DOMAIN_M * 0.06, C.DOMAIN_M * 0.06, 1000.0, 5.0, 1e-13, 0.0)
    q_far = fault_leakage_flux(pressure, saturation, far, cfg)
    ordered = q_far["q_fault_m3_s"] < q_near["q_fault_m3_s"]
    print(f"{'OK  ' if ordered else 'FAIL'} fault on the plume leaks more than one "
          f"outside it: {q_near['q_fault_m3_s']:.3e} vs {q_far['q_fault_m3_s']:.3e} "
          f"(distances {q_near['distance_plume_to_fault_m']:.0f} m vs "
          f"{q_far['distance_plume_to_fault_m']:.0f} m)")
    ok &= ordered

    # 6. Corey curve endpoints.
    krg_lo, krg_hi = corey_krg(cfg.corey_sgr, cfg), corey_krg(1.0 - cfg.corey_swr, cfg)
    endpoints = np.isclose(krg_lo, 0.0) and np.isclose(krg_hi, 1.0)
    print(f"{'OK  ' if endpoints else 'FAIL'} Corey endpoints: "
          f"krg(S_gr)={krg_lo:.4f}, krg(1-S_wr)={krg_hi:.4f}")
    ok &= endpoints

    return bool(ok)


if __name__ == "__main__":
    print(f"Caprock fracture pressure (assumed depth {C.RESERVOIR_DEPTH_M:.0f} m, "
          f"gradient {LEAKAGE.frac_gradient_bar_per_m} bar/m): "
          f"{LEAKAGE.frac_pressure_bar:.1f} bar")
    print(f"Initial reservoir pressure: {C.P_INIT_BAR} bar\n")
    print("Monotonicity checks on the T3 leakage model:")
    ok = check_monotonicity()
    print(f"\n{'ALL CHECKS PASSED' if ok else 'SOME CHECKS FAILED'}")
    raise SystemExit(0 if ok else 1)

In [ ]:
%%writefile src/leakage/features.py
"""Physics feature extraction: 128x128 fields -> a compact tabular row.

Two rules govern this module.

1. Never feed raw maps to XGBoost. Two 128x128 fields would be 32,768 columns,
   which destroys the interpretability that is the entire reason for using a
   gradient-boosted model here. We extract ~30 physically meaningful scalars
   instead.

2. The target is a FORECAST, not a description of the present. Each row carries
   features observed at timestep t and is labelled with the leakage flux at
   t + FORECAST_HORIZON (one full storage cycle, one year). Labelling a row with
   its own timestep's flux would let XGBoost rediscover the arithmetic of the
   label from `p_fault` and `s_fault` and score near-perfectly while predicting
   nothing. The forecast framing is what makes the number mean something.

Features can be built either from the simulator's own fields (for training) or
from U-Net predictions (for deployment, and to measure how much surrogate error
propagates into the risk estimate). `--source` selects which.
"""

from __future__ import annotations

from pathlib import Path

import numpy as np

from src import config as C
from src.config import LeakageConfig, LEAKAGE
from src.leakage.labels import (  # noqa: F401  (corey_krg re-exported for callers)
    Fault,
    boundary_ring_mask,
    caprock_margin,
    corey_krg,
    distance_to_fault_field,
    fault_cell_mask,
    fault_leakage_flux,
    h2_pore_volume,
    sample_faults,
)

# One storage cycle = 6 timesteps of 2 months = 1 year.
FORECAST_HORIZON = C.STEPS_PER_CYCLE

# Horizons emitted in a single pass, in timesteps (2 months each).
#
# The choice matters more than any hyperparameter, because the T3 label is a
# closed-form function of quantities that appear in the feature vector
# (fault permeability, area, and the pressure and saturation at the fault).
# At the same timestep the label is pure algebra. The ONLY genuine learning
# content is how the fields evolve between t and t+h, so the horizon decides
# how much of a task is left at all:
#
#     1   two months, part-cycle
#     3   half a cycle -> injection and withdrawal swap
#     6   one full cycle -> SAME phase, fields nearly repeat (near-trivial)
#    12   two cycles
#    30   five years -> genuine early warning, and where persistence should fail
#
# Emitting them together costs one pass, and lets the sweep in train_xgb show
# where the model stops being a restatement of the label formula.
FORECAST_HORIZONS = (1, 3, 6, 12, 30)


# --------------------------------------------------------------------------
# Per-timestep, fault-independent features
# --------------------------------------------------------------------------


def global_features(
    pressure_bar: np.ndarray,
    saturation: np.ndarray,
    porosity: np.ndarray,
    prev_pressure_bar: np.ndarray | None,
    prev_saturation: np.ndarray | None,
    cfg: LeakageConfig = LEAKAGE,
    ring: np.ndarray | None = None,
) -> dict[str, float]:
    """Pressure and plume descriptors that do not depend on any fault."""
    dt_years = C.MONTHS_PER_STEP / 12.0

    # -- pressure --
    gy, gx = np.gradient(pressure_bar.astype(np.float64), C.CELL_M)
    feats = {
        "p_max_bar": float(pressure_bar.max()),
        "p_min_bar": float(pressure_bar.min()),
        "p_mean_bar": float(pressure_bar.mean()),
        "p_p95_bar": float(np.percentile(pressure_bar, 95)),
        "p_well_bar": float(pressure_bar[C.WELL_BLOCK].mean()),
        "delta_p_bar": float(pressure_bar.max() - C.P_INIT_BAR),
        "p_grad_max_bar_per_m": float(np.hypot(gy, gx).max()),
    }
    feats["dp_dt_bar_per_year"] = (
        float((pressure_bar.mean() - prev_pressure_bar.mean()) / dt_years)
        if prev_pressure_bar is not None
        else 0.0
    )

    # -- plume --
    plume = saturation > cfg.plume_saturation_threshold
    hpv = h2_pore_volume(porosity, saturation)
    n_plume = int(plume.sum())
    feats.update(
        {
            "s_max": float(saturation.max()),
            "s_mean": float(saturation.mean()),
            "hpv_total_m3": float(hpv.sum()),
            "plume_area_m2": n_plume * C.CELL_AREA_M2,
            "plume_cell_count": float(n_plume),
        }
    )

    if n_plume:
        yy, xx = np.nonzero(plume)
        cy, cx = yy.mean(), xx.mean()
        feats["plume_centroid_offset_m"] = float(
            np.hypot(cy - C.GRID_CENTER, cx - C.GRID_CENTER) * C.CELL_M
        )
        feats["plume_max_radius_m"] = float(
            np.hypot(yy - C.GRID_CENTER, xx - C.GRID_CENTER).max() * C.CELL_M
        )
    else:
        feats["plume_centroid_offset_m"] = 0.0
        feats["plume_max_radius_m"] = 0.0

    if prev_saturation is not None:
        prev_plume = prev_saturation > cfg.plume_saturation_threshold
        if prev_plume.any():
            pyy, pxx = np.nonzero(prev_plume)
            prev_radius = np.hypot(pyy - C.GRID_CENTER, pxx - C.GRID_CENTER).max() * C.CELL_M
        else:
            prev_radius = 0.0
        feats["plume_front_speed_m_per_year"] = float(
            (feats["plume_max_radius_m"] - prev_radius) / dt_years
        )
        feats["hpv_rate_m3_per_year"] = float(
            (hpv.sum() - h2_pore_volume(porosity, prev_saturation).sum()) / dt_years
        )
    else:
        feats["plume_front_speed_m_per_year"] = 0.0
        feats["hpv_rate_m3_per_year"] = 0.0

    # -- T1 and T2, carried as features as well as secondary labels --
    ring = boundary_ring_mask() if ring is None else ring
    feats["boundary_saturation_max"] = float(saturation[ring].max())
    feats["boundary_hpv_fraction"] = float(
        hpv[ring].sum() / max(hpv.sum(), 1e-30)
    )
    feats["caprock_margin"] = caprock_margin(pressure_bar, cfg)["caprock_margin"]

    return feats


def geology_features(porosity: np.ndarray, permeability: np.ndarray) -> dict[str, float]:
    """Static descriptors of the realisation. Constant across a simulation's 60
    timesteps, but XGBoost has no notion of a simulation, so each row carries
    its own copy."""
    with np.errstate(divide="ignore"):
        logk = np.log10(np.maximum(permeability.astype(np.float64), 1e-30))
    return {
        "poro_mean": float(porosity.mean()),
        "poro_std": float(porosity.std()),
        "logk_mean": float(logk.mean()),
        "logk_std": float(logk.std()),
    }


def fault_features(
    pressure_bar: np.ndarray,
    saturation: np.ndarray,
    porosity: np.ndarray,
    permeability: np.ndarray,
    fault: Fault,
    distance_field: np.ndarray,
    mask: np.ndarray,
    cfg: LeakageConfig = LEAKAGE,
) -> dict[str, float]:
    """Fault-conditional features, plus the fault's own sampled properties.

    The sampled properties matter: without `fault_perm_m2` in the feature set
    the model cannot possibly separate "large plume on a sealed fault" from
    "small plume on an open one", and SHAP would attribute the difference to
    whatever proxy happens to correlate with it.
    """
    flux = fault_leakage_flux(
        pressure_bar, saturation, fault, cfg,
        fault_mask=mask, distance_field=distance_field,
    )
    with np.errstate(divide="ignore"):
        logk_fault = float(
            np.log10(np.maximum(permeability[mask].astype(np.float64), 1e-30)).mean()
        )
    d = flux["distance_plume_to_fault_m"]
    return {
        "fault_p_bar": flux["p_fault_bar"],
        "fault_s": flux["s_fault"],
        "fault_krg": flux["krg_fault"],
        "fault_overpressure_bar": flux["overpressure_bar"],
        "fault_delta_p_bar": flux["p_fault_bar"] - C.P_INIT_BAR,
        # Faults with no plume anywhere get the domain diagonal, a finite value
        # meaning "as far as it is possible to be", rather than NaN.
        "distance_plume_to_fault_m": float(d) if np.isfinite(d) else C.DOMAIN_M * 1.5,
        "fault_poro": float(porosity[mask].mean()),
        "fault_logk": logk_fault,
        "fault_log10_perm_m2": float(np.log10(fault.permeability_m2)),
        "fault_length_m": fault.length_m,
        "fault_width_m": fault.width_m,
        "fault_area_m2": fault.area_m2,
        "fault_radius_from_well_m": float(
            np.hypot(fault.x_m - C.DOMAIN_M / 2, fault.y_m - C.DOMAIN_M / 2)
        ),
        "_q_now_m3_s": flux["q_fault_m3_s"],  # label source; dropped from X
    }


def operational_features(t: int) -> dict[str, float]:
    return {
        "timestep": float(t),
        "cycle_number": float(C.cycle_number(t)),
        "cycle_index": float(C.cycle_index(t)),
        "step_in_cycle": float((t - 1) % C.STEPS_PER_CYCLE),
    }


# --------------------------------------------------------------------------
# Table construction
# --------------------------------------------------------------------------

# Columns that are labels or identifiers, never model inputs.
ID_COLUMNS = ("sim_id", "fault_id", "timestep_observed")
# Leaks the answer: the same physical quantity at the observed timestep. Kept in
# the table because the persistence baseline needs it, but never a model input.
EXCLUDED_FROM_X = ("_q_now_m3_s",)

LABEL_PREFIXES = ("q_future_h", "log_q_future_h", "leak_future_h")


def label_columns(h: int) -> dict[str, str]:
    return {"q": f"q_future_h{h}", "log_q": f"log_q_future_h{h}",
            "leak": f"leak_future_h{h}"}


def feature_columns(all_columns: list[str]) -> list[str]:
    """Model inputs: everything that is not an identifier, a label at ANY
    horizon, or the present-time flux."""
    drop = set(ID_COLUMNS) | set(EXCLUDED_FROM_X)
    return [
        c for c in all_columns
        if c not in drop and not any(c.startswith(p) for p in LABEL_PREFIXES)
    ]


def build_feature_table(
    constants: np.ndarray,
    states_pressure_bar,
    states_saturation,
    sim_ids: list[int],
    cfg: LeakageConfig = LEAKAGE,
    horizons: tuple[int, ...] = FORECAST_HORIZONS,
    n_faults: int | None = None,
    progress_every: int = 25,
    label_pressure_bar=None,
    label_saturation=None,
) -> tuple[np.ndarray, list[str]]:
    """Build the tabular dataset.

    `states_pressure_bar` and `states_saturation` are callables
    `(sim) -> (T, 128, 128)` in physical units, so the same code path serves
    both simulator fields and U-Net predictions.

    `label_pressure_bar` / `label_saturation` optionally supply a DIFFERENT
    field source for computing the labels. This matters:

      * Both from the simulator -> the training table.
      * Features from the U-Net, labels from the SIMULATOR -> the deployment
        question, "using surrogate-predicted fields, can we predict the TRUE
        leakage?" This is what measures how much surrogate error propagates
        into the risk estimate.

    Taking labels from the U-Net as well would only measure the surrogate's
    self-consistency: the errors would cancel and the score would look fine
    however wrong the fields were. Defaults to the feature source.

    Returns a float32 array plus its column names.
    """
    label_pressure_bar = label_pressure_bar or states_pressure_bar
    label_saturation = label_saturation or states_saturation
    labels_from_same_source = (
        label_pressure_bar is states_pressure_bar
        and label_saturation is states_saturation
    )
    n_faults = n_faults if n_faults is not None else cfg.n_faults_per_sim
    ring = boundary_ring_mask(C.GRID, cfg.boundary_ring_cells)

    rows: list[list[float]] = []
    columns: list[str] | None = None

    for n_done, sim in enumerate(sim_ids, start=1):
        poro = np.asarray(constants[sim, C.CONST_POROSITY], np.float32)
        perm = np.asarray(constants[sim, C.CONST_PERMEABILITY], np.float32)
        pressure = np.asarray(states_pressure_bar(sim), np.float32)   # (T, H, W)
        saturation = np.asarray(states_saturation(sim), np.float32)
        n_steps = pressure.shape[0]
        if labels_from_same_source:
            label_p, label_s = pressure, saturation
        else:
            label_p = np.asarray(label_pressure_bar(sim), np.float32)
            label_s = np.asarray(label_saturation(sim), np.float32)

        geo = geology_features(poro, perm)

        # Fault-independent features, once per timestep.
        glob = [
            global_features(
                pressure[i], saturation[i], poro,
                pressure[i - 1] if i > 0 else None,
                saturation[i - 1] if i > 0 else None,
                cfg, ring,
            )
            for i in range(n_steps)
        ]

        # Per-simulation fault realisations, seeded by sim so the whole table is
        # reproducible from the config alone.
        faults = sample_faults(n_faults, cfg, seed=cfg.rng_seed + sim)

        for fault in faults:
            dist = distance_to_fault_field(fault)
            mask = fault_cell_mask(dist)

            per_t = [
                fault_features(
                    pressure[i], saturation[i], poro, perm, fault, dist, mask, cfg
                )
                for i in range(n_steps)
            ]
            # Labels come from their own field source, which may differ.
            if labels_from_same_source:
                label_q = [p["_q_now_m3_s"] for p in per_t]
            else:
                label_q = [
                    fault_leakage_flux(
                        label_p[i], label_s[i], fault, cfg,
                        fault_mask=mask, distance_field=dist,
                    )["q_fault_m3_s"]
                    for i in range(n_steps)
                ]

            shortest = min(horizons)
            for i in range(n_steps - shortest):
                t = i + 1                      # timesteps are 1-based
                row = {
                    "sim_id": float(sim),
                    "fault_id": float(fault.fault_id),
                    "timestep_observed": float(t),
                    **glob[i],
                    **geo,
                    **per_t[i],
                    **operational_features(t),
                }
                # A label per horizon; NaN where t+h runs past the simulation.
                # train_xgb drops those rows for the horizon it selects, so each
                # horizon uses every row it legitimately can.
                for h in horizons:
                    j = i + h
                    names = label_columns(h)
                    if j < n_steps:
                        q = label_q[j]
                        row[names["q"]] = q
                        row[names["log_q"]] = float(np.log10(q + 1e-12))
                    else:
                        row[names["q"]] = np.nan
                        row[names["log_q"]] = np.nan
                    row[names["leak"]] = np.nan   # thresholded once the table exists

                if columns is None:
                    columns = list(row.keys())
                rows.append([row[c] for c in columns])

        if progress_every and n_done % progress_every == 0:
            print(
                f"  {n_done}/{len(sim_ids)} simulations, {len(rows):,} rows",
                flush=True,
            )

    table = np.asarray(rows, dtype=np.float32)
    return table, (columns or [])


def apply_leak_threshold(
    table: np.ndarray,
    columns: list[str],
    quantile: float = 0.90,
    horizons: tuple[int, ...] = FORECAST_HORIZONS,
) -> tuple[np.ndarray, dict[int, float]]:
    """Binarise the forecast target, once per horizon.

    The threshold is a quantile of the NON-ZERO fluxes rather than an absolute
    m^3/s value, because the absolute scale is set by the assumed fault
    permeability and caprock thickness and so carries no independent meaning.
    Reported as a concrete flux alongside every result.

    Each horizon gets its OWN threshold from its own flux distribution, so the
    positive-class rate stays comparable across the sweep and differences in
    score reflect difficulty rather than shifting class balance.
    """
    thresholds: dict[int, float] = {}
    for h in horizons:
        names = label_columns(h)
        qi, li = columns.index(names["q"]), columns.index(names["leak"])
        q = table[:, qi]
        valid = np.isfinite(q)
        positive = q[valid & (q > 0)]
        thr = float(np.quantile(positive, quantile)) if positive.size else 0.0
        thresholds[h] = thr
        table[:, li] = np.where(valid, (q > thr).astype(np.float32), np.nan)
    return table, thresholds


def save_table(path: Path, table: np.ndarray, columns: list[str], meta: dict) -> None:
    import json

    path.parent.mkdir(parents=True, exist_ok=True)
    np.save(path, table)
    path.with_suffix(".columns.json").write_text(
        json.dumps({"columns": columns, "meta": meta}, indent=2)
    )


def load_table(path: Path) -> tuple[np.ndarray, list[str], dict]:
    import json

    table = np.load(path)
    payload = json.loads(Path(path).with_suffix(".columns.json").read_text())
    return table, payload["columns"], payload.get("meta", {})

In [ ]:
# Per-simulation geology + caprock-margin features, using the real
# src.leakage functions -- no reimplementation.
import sys
sys.path.insert(0, "/kaggle/working")

import numpy as np
import pandas as pd

from src import config as C
from src.config import LEAKAGE
from src.leakage.labels import caprock_margin
from src.leakage.features import geology_features

constants = np.load(DATA / "constants.npy", mmap_mode="r")   # (1000, 2, 128, 128) float32
states = np.load(DATA / "states.npy", mmap_mode="r")         # (1000, 60, 2, 128, 128) float16
n_sims = constants.shape[0]
print(f"{n_sims} simulations")

rows = []
for sim in range(n_sims):
    poro = np.asarray(constants[sim, C.CONST_POROSITY], np.float32)
    perm = np.asarray(constants[sim, C.CONST_PERMEABILITY], np.float32)
    geo = geology_features(poro, perm)

    # states stores pressure CENTRED (P_bar - P_INIT_BAR); the max centred
    # value across all 60 timesteps is exactly (P_max_bar - P_INIT_BAR), so
    # this is the peak pressure ever reached anywhere in this realisation.
    p_centred_peak = float(np.asarray(states[sim, :, C.STATE_PRESSURE], np.float32).max())
    p_max_bar = p_centred_peak + C.P_INIT_BAR
    margin = caprock_margin(np.array([p_max_bar], dtype=np.float32), LEAKAGE)

    rows.append({
        "sim_id": sim,
        **geo,
        "p_max_bar": margin["p_max_bar"],
        "caprock_margin_peak": margin["caprock_margin"],
    })
    if (sim + 1) % 200 == 0:
        print(f"  {sim + 1}/{n_sims}")

df = pd.DataFrame(rows)
df.to_csv("/kaggle/working/site_features.csv", index=False)
print(df.describe())

In [ ]:
# Cluster the 1000 realisations on geology + caprock margin only --
# deliberately NOT mixing in the fault-conditional leakage-risk model, so
# this stays a pure geological-suitability screen (module 1), separate from
# the fault-hypothesis risk model (module 2).
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

FEATURE_COLS = ["poro_mean", "poro_std", "logk_mean", "logk_std", "caprock_margin_peak"]
X = StandardScaler().fit_transform(df[FEATURE_COLS].values)

best_k, best_score, best_model = None, -1.0, None
for k in range(2, 7):
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X)
    score = silhouette_score(X, km.labels_)
    print(f"k={k}  silhouette={score:.4f}")
    if score > best_score:
        best_k, best_score, best_model = k, score, km

print(f"\nChosen k={best_k} (silhouette {best_score:.4f})")
df["cluster"] = best_model.labels_

summary = df.groupby("cluster")[FEATURE_COLS].agg(["mean", "count"])
print(summary)

df.to_csv("/kaggle/working/site_clusters.csv", index=False)
summary.to_csv("/kaggle/working/cluster_summary.csv")
print("\nwrote site_clusters.csv, cluster_summary.csv")

## Before you close the session

Save Version -> **Quick Save** to persist `site_clusters.csv` and
`cluster_summary.csv` as this kernel's output — small files, no large data to
keep around this time.